### Reddit Subreddit Rules Evolution for Health Crises

#### Overview
This module collects historical moderation rules from health-related subreddits to study how rules evolved over time during public health crises. The focus is on identifying when rules were introduced, modified, or removed, and how these changes aligned with major events such as the COVID-19 pandemic (2020–2023) and the mpox outbreak (2022).

#### Data Collected
The data is limited to:
- Subreddit rules and guidelines  
- Pinned moderator announcements  
- Policy clarifications issued by moderators  
- Timestamps of rule updates  

#### Objective
The goal is to construct a temporal map of rule evolution to analyze how subreddit governance adapted in response to health misinformation and shifting public health contexts.


In [10]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import json
# Function to save data periodically
def periodicSave(data, description):
    data = pd.DataFrame(data)
    filename = f"./rules{description}.json"

    # Save as JSON (append-friendly)
    with open(filename, "a", encoding="utf-8") as f:
        json.dump(data.to_dict(orient="records"), f, ensure_ascii=False)
        f.write("\n")  # newline separated JSON objects

    print(f"Periodic save completed. Total posts saved: {len(data)}")



#### Collecting Data

In [11]:
import urllib.request
from bs4 import BeautifulSoup


RULE_HEADER_SELECTOR = "h1[id^='wiki_rule_'], h2[id^='wiki_rule_'], h3[id^='wiki_rule_']"

def extractRulesFromLink(pageUrl):
    # Fetch HTML
    req = urllib.request.Request(
        pageUrl,
        headers={"User-Agent": "Mozilla/5.0 (academic research bot)"}
    )
    with urllib.request.urlopen(req) as response:
        html = response.read()

    soup = BeautifulSoup(html, "html.parser")

    rules = []

    rule_headers = soup.select(RULE_HEADER_SELECTOR)

    for header in rule_headers:
        title = header.get_text(strip=True)

        description_parts = []
        sibling = header.find_next_sibling()

        # Stop at the next rule header (h1/h2/h3 with wiki_rule_)
        while sibling and not (
            sibling.name in ["h1", "h2", "h3"]
            and sibling.get("id", "").startswith("wiki_rule_")
        ):
            if sibling.name in ["p", "ul", "ol"]:
                description_parts.append(
                    sibling.get_text(" ", strip=True)
                )
            sibling = sibling.find_next_sibling()

        description = " ".join(description_parts)

        rules.append({
            "title": title,
            "description": description,
        })

    return rules


In [14]:
import pandas as pd
import time

data = pd.read_json("./data/dateSnapshots/dateSnapshots2023.json")

MAX_RETRIES = 5
RETRY_DELAY = 10  # seconds

for index in range(len(data)):
    link = data.loc[index, "SnapshotLink"]
    snapshot_date = data.loc[index, "SnapshotDate"]

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            rules = extractRulesFromLink(link)
            rules = pd.DataFrame(rules)

            rules.to_json(
                f"./data/dateSnapshots/rules/rules{snapshot_date}.json"
            )

            print(f"[{index}] Link processed ✅ ({link})")
            break  # success → exit retry loop

        except Exception as e:
            print(
                f"[{index}] Attempt {attempt}/{MAX_RETRIES} failed ❌ "
                f"({link})"
            )

            if attempt == MAX_RETRIES:
                print(f"[{index}] Giving up on link ⛔\n")
            else:
                time.sleep(RETRY_DELAY)


[0] Link processed ✅ (https://web.archive.org/web/20230224/https://www.reddit.com/r/Coronavirus/wiki/rules/)
[1] Link processed ✅ (https://web.archive.org/web/20230322/https://www.reddit.com/r/Coronavirus/wiki/rules/)
[2] Attempt 1/5 failed ❌ (https://web.archive.org/web/20230325/https://www.reddit.com/r/Coronavirus/wiki/rules/)
[2] Attempt 2/5 failed ❌ (https://web.archive.org/web/20230325/https://www.reddit.com/r/Coronavirus/wiki/rules/)
[2] Link processed ✅ (https://web.archive.org/web/20230325/https://www.reddit.com/r/Coronavirus/wiki/rules/)
[3] Link processed ✅ (https://web.archive.org/web/20230329/https://www.reddit.com/r/Coronavirus/wiki/rules/)
[4] Link processed ✅ (https://web.archive.org/web/20230405/https://www.reddit.com/r/Coronavirus/wiki/rules/)
[5] Link processed ✅ (https://web.archive.org/web/20230419/https://www.reddit.com/r/Coronavirus/wiki/rules/)
[6] Attempt 1/5 failed ❌ (https://web.archive.org/web/20230424/https://www.reddit.com/r/Coronavirus/wiki/rules/)
[6] Lin